[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# The Native Interfaces


## What you will be able to do

Train LightGBM and XGBoost through their own interfaces, from a `Dataset` or a `DMatrix` and a
dictionary of settings, record the validation loss round by round, and reach the booster inside a
scikit-learn model.

## The idea

### The problem

The scikit-learn classes in the **scikit-learn Interface** notebook cover most of what a project
needs, and both libraries were written before them. Their documentation, their examples and much of
the code you will read train a model a different way, with the data in an object of the library's
own, the settings in a dictionary, and a function called `train` that returns the model. Code written
that way often records the loss on a validation set after every round, which the **Learning Rate and
Trees** notebook reads to choose a number of trees, and reading it needs the native interface.

### A formal definition

> A library's **native interface** is the way of training it that the library itself defines. In
> LightGBM, `lgb.Dataset` holds the table and its answers, and `lgb.train(params, dataset,
> num_boost_round)` returns a `Booster`. In XGBoost, `xgb.DMatrix` holds them, and
> `xgb.train(params, dtrain, num_boost_round)` returns a `Booster`. `params` is a dictionary of
> settings by name.

### Why it works that way

A `Dataset` or a `DMatrix` is the table as the library wants it, prepared once so that every round of
training reads it without converting it again. LightGBM sorts a `Dataset`'s columns into bins when
training first uses it. A `DMatrix` holds the values themselves, with categories coded, and XGBoost
sorts them into bins as training starts. A scikit-learn class builds the `Dataset` or `DMatrix`
inside `fit` and hides it. The booster it trains is the same object the native `train` returns, and
predicts the same numbers.

The dictionary is looser than a constructor. A setting can be spelled several ways, since LightGBM
accepts aliases such as `eta` for `learning_rate` and `num_iterations` for the number of rounds, and
XGBoost accepts `eta` and `learning_rate` alike. A name a library does not know is passed along
anyway, and XGBoost warns that it was not used. The two libraries also name the same idea
differently. Squared error is the objective `"regression"` in LightGBM and `"reg:squarederror"` in XGBoost, and the loss it reports is `l2`, the
mean squared error, in LightGBM and `rmse`, its square root, in XGBoost.

XGBoost's `Booster.predict` takes only a `DMatrix`, and `Booster.inplace_predict` takes a pandas
table or an array as it is. LightGBM's `predict` takes either as it is.

### Where this shows up

The **Early Stopping** notebook uses the recorded losses to stop training, and the **Saving and
Loading Boosters** notebook saves the booster both interfaces return.

### What this notebook covers

- `lgb.Dataset` and `lgb.train`, and what the `Dataset` found in the table
- `xgb.DMatrix` and `xgb.train`
- the same settings under the names each library gives them
- the loss on the validation stations, recorded after every round
- the booster inside a scikit-learn model
- four mistakes with the dictionary and the data

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import lightgbm as lgb

x = np.arange(100.0).reshape(-1, 1)
dataset = lgb.Dataset(x, label=2 * x.ravel())
booster = lgb.train({"objective": "regression", "verbose": -1}, dataset, num_boost_round=20)
print(type(booster).__name__, booster.num_trees(), booster.predict(np.array([[50.0]])).round(1))
```

```
Booster 20 [100.6]
```

The `Dataset` holds the column of numbers and its answers, each twice the number. `lgb.train` takes
the settings as a dictionary and returns a `Booster` of 20 trees, which predicts 100.6 for 50, where
the answer is 100.

## Setup

The module that catches warnings, two libraries for the table, the two boosting libraries, and a
score.

- `warnings` catches XGBoost's warning about an unused setting in the Common errors
- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are the two libraries, and the cell prints their versions
- `mean_squared_error` scores the boosters on the test stations

`station_days` makes the same 20,000 made-up station-days as in the **Why Gradient Boosting**
notebook. Each row is one day at one of 400 weather stations in Norway, with the station's latitude,
elevation, distance from the coast and terrain, the day's cloud, wind and snow, and `tmin`, the day's
lowest temperature. A generator started from a fixed seed draws them, so they are the same on every
run. The cell splits the stations into 300 for training, 50 for validation and 50 for testing,
keeping each station's days together. `FEATURES` names the eight columns a model is given.

In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### LightGBM's Dataset and train

The settings are LightGBM's defaults written out, with `verbose` at -1 to keep its log quiet and a
`seed`.

In [2]:
y_train, y_test = train["tmin"], test["tmin"]

lightgbm_params = {"objective": "regression", "learning_rate": 0.1, "num_leaves": 31, "seed": 0, "verbose": -1}
train_dataset = lgb.Dataset(train[FEATURES], label=y_train)
lightgbm_booster = lgb.train(lightgbm_params, train_dataset, num_boost_round=100)

print(type(lightgbm_booster).__name__, lightgbm_booster.num_trees(), "trees")
print("test loss:", round(mean_squared_error(y_test, lightgbm_booster.predict(test[FEATURES])), 3))

Booster 100 trees
test loss: 2.555


The booster has 100 trees and a test loss of 2.555, the same as `LGBMRegressor` with its defaults in
the **scikit-learn Interface** notebook, since these settings are the ones that class uses.

### What the Dataset found

The `Dataset` read the column names and the category from the pandas table, and the booster records
both, with the settings it trained with.

In [3]:
print(lightgbm_booster.feature_name())
print(train_dataset.pandas_categorical)
print(lightgbm_booster.params)

['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm']
[['coastal', 'fjord', 'inland', 'valley', 'mountain']]
{'objective': 'regression', 'learning_rate': 0.1, 'num_leaves': 31, 'seed': 0, 'verbose': -1, 'num_iterations': 100, 'categorical_column': [3]}


The booster kept the eight column names, and the `Dataset` found the categories of `terrain`, in
their order. `params` holds the settings the booster trained with, and two that LightGBM added:
`num_iterations`, the 100 rounds asked for, and `categorical_column`, the position of `terrain`, 3,
counting from 0.

### XGBoost's DMatrix and train

The cell trains the same model in XGBoost's terms, with XGBoost's defaults written out: an objective
of `reg:squarederror`, a learning rate of 0.3 and a depth of 6.

In [4]:
xgboost_params = {"objective": "reg:squarederror", "learning_rate": 0.3, "max_depth": 6, "seed": 0}
dtrain = xgb.DMatrix(train[FEATURES], label=y_train)
dtest = xgb.DMatrix(test[FEATURES], label=y_test)
xgboost_booster = xgb.train(xgboost_params, dtrain, num_boost_round=100)

print(type(xgboost_booster).__name__, xgboost_booster.num_boosted_rounds(), "rounds")
print("test loss:", round(mean_squared_error(y_test, xgboost_booster.predict(dtest)), 3))
print(xgboost_booster.feature_names, xgboost_booster.feature_types)

Booster 100 rounds
test loss: 2.883
['latitude', 'elevation_m', 'coast_km', 'terrain', 'day_of_year', 'cloud_cover', 'wind_ms', 'snow_cm'] ['float', 'int', 'float', 'c', 'int', 'float', 'float', 'float']


The booster has 100 rounds and a test loss of 2.883, the same as `XGBRegressor` with its defaults in
the **scikit-learn Interface** notebook. `feature_types` shows what XGBoost made of each column, with
`c` marking `terrain` as a category. XGBoost 3.4 reads a pandas category without being asked, so this
`DMatrix` needed no `enable_categorical=True`, which older code passes.

### A setting under more than one name

LightGBM reads `eta` as another name for `learning_rate`. Two boosters trained with the rate under
each name make the same predictions.

In [5]:
by_eta = lgb.train({"objective": "regression", "eta": 0.3, "seed": 0, "verbose": -1}, train_dataset, 20)
by_rate = lgb.train({"objective": "regression", "learning_rate": 0.3, "seed": 0, "verbose": -1}, train_dataset, 20)
print("same predictions:", np.array_equal(by_eta.predict(test[FEATURES]), by_rate.predict(test[FEATURES])))

same predictions: True


The two boosters agree on every prediction, so the two names set one thing. The table lists the
names this guide uses.

| What it sets | LightGBM | XGBoost |
|---|---|---|
| the share of each tree added | `learning_rate`<br>`eta` | `learning_rate`<br>`eta` |
| the number of rounds | `num_boost_round`, the argument<br>`num_iterations` and other aliases in `params` | `num_boost_round`, the argument |
| the size of a tree | `num_leaves` | `max_depth` |
| a penalty on leaf values | `lambda_l2`<br>`reg_lambda` | `lambda`<br>`reg_lambda` |
| the seed | `seed` | `seed` |
| the loss for squared error | `"regression"`, reported as `l2` | `"reg:squarederror"`, reported as `rmse` |

### The validation loss after every round

LightGBM takes the validation table as a second `Dataset`, with `reference=train_dataset` so that it
uses the same bins, and `record_evaluation` fills a dictionary with the loss after each round. XGBoost
takes a list of `(DMatrix, name)` pairs and fills `evals_result`, and `verbose_eval=False` stops it
printing a line per round.

In [6]:
valid_dataset = lgb.Dataset(valid[FEATURES], label=valid["tmin"], reference=train_dataset)
lightgbm_losses = {}
lgb.train(lightgbm_params, train_dataset, num_boost_round=300, valid_sets=[valid_dataset], valid_names=["valid"],
          callbacks=[lgb.record_evaluation(lightgbm_losses)])

dvalid = xgb.DMatrix(valid[FEATURES], label=valid["tmin"])
xgboost_losses = {}
xgb.train(xgboost_params, dtrain, num_boost_round=300, evals=[(dvalid, "valid")], evals_result=xgboost_losses,
          verbose_eval=False)

for round_number in (10, 50, 100, 200, 300):
    l2 = lightgbm_losses["valid"]["l2"][round_number - 1]
    rmse = xgboost_losses["valid"]["rmse"][round_number - 1]
    print(round_number, "| LightGBM l2", round(l2, 3), "| XGBoost rmse", round(rmse, 3), "squared", round(rmse ** 2, 3))

10 | LightGBM l2 10.156 | XGBoost rmse 1.849 squared 3.419
50 | LightGBM l2 2.724 | XGBoost rmse 1.707 squared 2.913
100 | LightGBM l2 2.553 | XGBoost rmse 1.71 squared 2.924
200 | LightGBM l2 2.581 | XGBoost rmse 1.732 squared 2.999
300 | LightGBM l2 2.58 | XGBoost rmse 1.739 squared 3.023


LightGBM's `l2` is the mean squared error and XGBoost's `rmse` its square root, so the last column
squares it to compare the two. Of the rounds printed, LightGBM's validation loss is lowest at round
100, 2.553, and XGBoost's, at a learning rate three times as large, at round 50, 2.913. At every
round printed after those, both losses are higher, which is the trade the **Learning Rate and Trees**
notebook measures.

### The booster inside a scikit-learn model

`booster_` is the LightGBM booster a scikit-learn model trained, and `get_booster()` the XGBoost one.
Each predicts what its model predicts.

In [7]:
lightgbm_model = lgb.LGBMRegressor(random_state=0, verbose=-1).fit(train[FEATURES], y_train)
xgboost_model = xgb.XGBRegressor(random_state=0).fit(train[FEATURES], y_train)

print(np.array_equal(lightgbm_model.booster_.predict(test[FEATURES]), lightgbm_model.predict(test[FEATURES])))
print(np.array_equal(xgboost_model.get_booster().predict(dtest), xgboost_model.predict(test[FEATURES])))
print("same as the native boosters:",
      np.array_equal(lightgbm_model.predict(test[FEATURES]), lightgbm_booster.predict(test[FEATURES])),
      np.array_equal(xgboost_model.predict(test[FEATURES]), xgboost_booster.predict(dtest)))

True
True
same as the native boosters: True True


Each booster gives exactly its model's predictions, and both models give exactly the predictions of
the native boosters trained with the same settings.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/03-the-native-interfaces-solutions.ipynb).

**1.** Train a LightGBM booster with `lgb.train`, `num_leaves` of 15 and a learning rate of 0.05, for 300 rounds, and print its validation loss.

In [8]:
# your code here

**2.** Train an XGBoost booster with `xgb.train`, a `max_depth` of 4 and a learning rate of 0.1, for 300 rounds, and print its validation loss.

In [9]:
# your code here

**3.** Record LightGBM's validation loss for 300 rounds at a learning rate of 0.1, as the worked examples do, and print the round with the lowest loss and that loss.

In [10]:
# your code here

**4.** Fit `LGBMRegressor(n_estimators=200, num_leaves=15, random_state=0, verbose=-1)`, and print the number of leaves, the learning rate and the number of rounds in its `booster_.params`, and the number of trees the booster has.

In [11]:
# your code here

**5.** Give `xgb.train` both the training and the validation `DMatrix` in `evals`, and print the last round's `rmse` for each.

In [12]:
# your code here

## Common errors

### `TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'pandas.core.frame.DataFrame'>)`

In [13]:
xgboost_booster.predict(test[FEATURES])

TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'pandas.core.frame.DataFrame'>)

`Booster.predict` takes only a `DMatrix`, and its message is a tuple of the text and the class it was
given. Wrap the table first, as the worked examples do with `dtest`, or call `inplace_predict`, which
takes the table as it is and is what the scikit-learn model's `predict` uses.

In [14]:
print(np.array_equal(xgboost_booster.inplace_predict(test[FEATURES]), xgboost_booster.predict(dtest)))

True


`inplace_predict` on the table gives exactly the predictions of `predict` on `dtest`.

### `LightGBMError: Unknown objective type name: reg:squarederror`

In [15]:
lgb.train({"objective": "reg:squarederror", "verbose": -1}, train_dataset, num_boost_round=5)

[LightGBM] [Fatal] Unknown objective type name: reg:squarederror


LightGBMError: Unknown objective type name: reg:squarederror

LightGBM prints the message as a `[Fatal]` line, then raises it. `reg:squarederror` is XGBoost's
name for squared error, and LightGBM's is `regression`. XGBoost given LightGBM's name raises too, with
a time, a path inside its build and a list of every objective it knows, so the cell below prints only
its first sentence.

In [16]:
try:
    xgb.train({"objective": "regression"}, dtrain, num_boost_round=5)
except xgb.core.XGBoostError as error:
    print("XGBoostError:", str(error).splitlines()[0].split(": ", 1)[1])

XGBoostError: Unknown objective function: `regression`


Use each library's own name for an objective, `regression` in LightGBM and `reg:squarederror` in
XGBoost.

### `UserWarning: Parameters: { "num_leaves" } are not used.`

In [17]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    leafy_booster = xgb.train({**xgboost_params, "num_leaves": 15}, dtrain, num_boost_round=5)
for warning in caught:
    message = str(warning.message)
    print(f"{warning.category.__name__}: {message[message.index('Parameters'):].strip()}")
print(leafy_booster.num_boosted_rounds(), "rounds trained")

5 rounds trained


XGBoost trained its five rounds and ignored `num_leaves`, a LightGBM setting. Its warning begins with
the time and a path inside XGBoost's build, which the cell leaves out. XGBoost's nearest setting is
`max_leaves`, and the **Tree Shape** notebook compares how the two libraries limit the size of a tree.

### No error, and seven trees where five were asked for

In [18]:
model = lgb.LGBMRegressor(n_estimators=5, num_iterations=7, random_state=0, verbose=-1)
model.fit(train[FEATURES], y_train)
print(model.get_params()["n_estimators"], "trees asked for,", model.booster_.num_trees(), "trained")

5 trees asked for, 7 trained


`num_iterations` is another of LightGBM's names for the number of rounds, and given beside
`n_estimators` it wins. LightGBM gives no warning, with `verbose` at -1 or at its default, so give
each setting once, under one name.

## Recap

- You can train a LightGBM booster from a `Dataset` and an XGBoost booster from a `DMatrix`, with
  the settings in a dictionary.
- You can find a setting under the name each library gives it, and an objective under each library's
  own name.
- You can record the validation loss after every round in both libraries.
- You can reach the booster inside a scikit-learn model, and predict with an XGBoost booster from a
  `DMatrix`.

## What is next

The **Learning Rate and Trees** notebook uses the recorded validation loss to weigh the learning rate
against the number of trees.

---

&#8592; **Previous:** [The scikit-learn Interface](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/02-the-scikit-learn-interface.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Learning Rate and Trees](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/04-learning-rate-and-trees.ipynb) &#8594;
